# 02 — Feature Engineering

**InternHub — IT Interview Knowledge Tracing System**

Pipeline:
1. Load interaction_log.csv đã fix
2. Tạo labels: binary (AUC) + normalized quality (RMSE)
3. Encode KC (knowledge component) và user
4. Build sequences per user cho DKT/SAKT
5. Build cold-start features từ self_rating_log
6. Train/Val/Test split by user (70/15/15)
7. Save → data/processed/

## 0. Setup

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

ROOT      = Path('..')
RAW       = ROOT / 'data' / 'raw' / 'simulation'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Setup OK — processed dir:', PROCESSED)

---
## 1. Load & Kiểm tra dữ liệu

In [ ]:
log = pd.read_csv(RAW / 'interaction_log.csv')
log['timestamp'] = pd.to_datetime(log['timestamp'], errors='coerce')

print(f'Shape: {log.shape}')
print(f'Users: {log["user_id"].nunique()}')
print(f'Questions: {log["question_id"].nunique()}')
print(f'Competencies: {sorted(log["competency"].unique().tolist())}')
print()

q_pct = log['quality_score'].value_counts(normalize=True).sort_index() * 100
print('Quality distribution:')
for k, v in q_pct.items():
    print(f'  {k}: {v:.1f}%')

seq_len = log.groupby('user_id').size()
print(f'\nSeq length: min={seq_len.min()}, max={seq_len.max()}, mean={seq_len.mean():.1f}, std={seq_len.std():.1f}')

---
## 2. Labels

| Label | Công thức | Dùng cho |
|---|---|---|
| `correct` | `quality_score >= 1` → {0,1} | AUC (BKT, DKT-Binary, SAKT-Binary) |
| `quality_norm` | `quality_score / 2.0` → {0.0, 0.5, 1.0} | RMSE (DKT-Quality, SAKT-Quality) |

In [ ]:
log['correct']      = (log['quality_score'] >= 1).astype(int)
log['quality_norm'] = log['quality_score'] / 2.0

print('correct distribution:')
print(log['correct'].value_counts(normalize=True).round(3))
print()
print('quality_norm values:', sorted(log['quality_norm'].unique().tolist()))

---
## 3. Encoding

In [ ]:
# ── User encoding ─────────────────────────────────────────────────────────
user_ids   = sorted(log['user_id'].unique())
user2idx   = {uid: i for i, uid in enumerate(user_ids)}
log['user_idx'] = log['user_id'].map(user2idx)

# ── KC (competency) encoding ──────────────────────────────────────────────
kc_ids     = sorted(log['competency'].unique())
kc2idx     = {kc: i for i, kc in enumerate(kc_ids)}
log['kc_idx'] = log['competency'].map(kc2idx)

# ── Question encoding ─────────────────────────────────────────────────────
q_ids      = sorted(log['question_id'].unique())
q2idx      = {qid: i for i, qid in enumerate(q_ids)}
log['q_idx'] = log['question_id'].map(q2idx)

# ── Difficulty encoding ───────────────────────────────────────────────────
diff_map   = {'EASY': 0, 'MEDIUM': 1, 'HARD': 2}
log['diff_idx'] = log['difficulty_label'].map(diff_map).fillna(1).astype(int)

n_users = len(user_ids)
n_kcs   = len(kc_ids)
n_qs    = len(q_ids)

print(f'n_users={n_users}, n_kcs={n_kcs}, n_questions={n_qs}')
print('KC mapping:')
for kc, idx in kc2idx.items():
    print(f'  {idx}: {kc}')

---
## 4. Train / Val / Test Split (by user)

Split **theo user** (không theo time) để tránh data leakage:
- Train: 70% users
- Val:   15% users  
- Test:  15% users

In [ ]:
rng = np.random.default_rng(SEED)
all_users  = np.array(user_ids)
rng.shuffle(all_users)

n_train = int(0.70 * n_users)
n_val   = int(0.15 * n_users)

train_users = set(all_users[:n_train])
val_users   = set(all_users[n_train:n_train + n_val])
test_users  = set(all_users[n_train + n_val:])

log['split'] = log['user_id'].apply(
    lambda u: 'train' if u in train_users else ('val' if u in val_users else 'test')
)

split_stats = log.groupby('split').agg(
    n_users=('user_id', 'nunique'),
    n_rows=('user_id', 'count'),
    correct_rate=('correct', 'mean'),
).round(3)
print(split_stats)

---
## 5. Build Sequences (cho DKT / SAKT)

In [ ]:
def build_sequences(df: pd.DataFrame) -> list:
    """
    Mỗi user → 1 dict chứa sequence theo session_order.
    Trả về list of dicts.
    """
    seqs = []
    for uid, grp in df.sort_values('session_order').groupby('user_id'):
        seqs.append({
            'user_id':      uid,
            'user_idx':     grp['user_idx'].iloc[0],
            'kc_seq':       grp['kc_idx'].tolist(),          # KC index
            'q_seq':        grp['q_idx'].tolist(),            # Question index
            'correct_seq':  grp['correct'].tolist(),          # Binary label
            'quality_seq':  grp['quality_norm'].tolist(),     # Float label
            'diff_seq':     grp['diff_idx'].tolist(),         # Difficulty
            'skill_seq':    grp['skill_before'].tolist(),     # True skill (for analysis)
            'seq_len':      len(grp),
            'split':        grp['split'].iloc[0],
        })
    return seqs

all_seqs   = build_sequences(log)
train_seqs = [s for s in all_seqs if s['split'] == 'train']
val_seqs   = [s for s in all_seqs if s['split'] == 'val']
test_seqs  = [s for s in all_seqs if s['split'] == 'test']

print(f'train_seqs: {len(train_seqs)}')
print(f'val_seqs:   {len(val_seqs)}')
print(f'test_seqs:  {len(test_seqs)}')
print(f'Sample seq keys: {list(train_seqs[0].keys())}')
print(f'Sample seq_len: {train_seqs[0]["seq_len"]}')
print(f'Sample kc_seq[:5]: {train_seqs[0]["kc_seq"][:5]}')
print(f'Sample correct_seq[:5]: {train_seqs[0]["correct_seq"][:5]}')

In [ ]:
# Phân tích seq length distribution
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

lens = [s['seq_len'] for s in all_seqs]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lens, bins=25, color='#00e5ff', edgecolor='white')
ax.axvline(np.mean(lens), color='red', linestyle='--', label=f'mean={np.mean(lens):.1f}')
ax.set_title('Sequence Length Distribution (after fix)')
ax.set_xlabel('# Interactions per User')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()
print(f'std={np.std(lens):.1f}, min={min(lens)}, max={max(lens)}')

---
## 6. Cold-Start Features (từ self_rating_log)

In [ ]:
sr = pd.read_csv(RAW / 'self_rating_log.csv')
print(f'self_rating shape: {sr.shape}')
print(f'Columns: {sr.columns.tolist()}')
sr.head(3)

In [ ]:
# Normalize self_rating per user → cold-start skill vector
# Mục tiêu: 1 vector [0,1] per user covering các competency

# Tìm rating column và competency column
rating_col = next((c for c in ['self_rating_raw', 'self_rating', 'rating', 'perceived_skill'] if c in sr.columns), None)
comp_col   = next((c for c in ['competency', 'skill', 'skill_group'] if c in sr.columns), None)
uid_col    = next((c for c in ['user_id', 'userId'] if c in sr.columns), None)

print(f'uid_col={uid_col}, comp_col={comp_col}, rating_col={rating_col}')

if rating_col and comp_col and uid_col:
    # Normalize: nếu rating 1-5 → /5, nếu 0-1 giữ nguyên
    max_rating = sr[rating_col].max()
    if max_rating > 1:
        sr['rating_norm'] = sr[rating_col] / max_rating
    else:
        sr['rating_norm'] = sr[rating_col]

    # Normalize competency name
    sr['comp_norm'] = sr[comp_col].str.strip().str.lower()

    # Pivot: user × competency → avg self_rating
    cs_pivot = sr.groupby([uid_col, 'comp_norm'])['rating_norm'].mean().unstack(fill_value=0.0)
    print(f'Cold-start matrix: {cs_pivot.shape}')
    print(f'Competency columns: {cs_pivot.columns.tolist()}')
    cs_pivot.head(3)
else:
    print('Columns không khớp — xem schema bên dưới')
    print(sr.describe())

In [ ]:
# Map cold-start vector vào kc_ids (padding 0 nếu không có)
if 'cs_pivot' in dir():
    coldstart = {}
    for uid in user_ids:
        if uid in cs_pivot.index:
            row = cs_pivot.loc[uid]
            vec = [float(row.get(kc, 0.0)) for kc in kc_ids]
        else:
            vec = [0.5] * n_kcs  # fallback: uniform prior
        coldstart[uid] = vec

    print(f'Cold-start vectors built for {len(coldstart)} users')
    sample_uid = user_ids[0]
    print(f'Sample ({sample_uid}): {[round(v,3) for v in coldstart[sample_uid]]}')
else:
    # Nếu self_rating schema khác: dùng initial skill_before làm fallback
    coldstart = {}
    for uid, grp in log.groupby('user_id'):
        first_skill = grp.sort_values('session_order').head(5)['skill_before'].mean()
        coldstart[uid] = [first_skill] * n_kcs
    print(f'Cold-start (fallback from skill_before): {len(coldstart)} users')

---
## 7. Save processed data

In [ ]:
# ── Metadata ──────────────────────────────────────────────────────────────
metadata = {
    'n_users':    n_users,
    'n_kcs':      n_kcs,
    'n_questions': n_qs,
    'kc_ids':     kc_ids,
    'kc2idx':     kc2idx,
    'user2idx':   user2idx,
    'q2idx':      q2idx,
    'train_users': list(train_users),
    'val_users':   list(val_users),
    'test_users':  list(test_users),
}

# ── Save ──────────────────────────────────────────────────────────────────
with open(PROCESSED / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

with open(PROCESSED / 'train_seqs.pkl', 'wb') as f:
    pickle.dump(train_seqs, f)

with open(PROCESSED / 'val_seqs.pkl', 'wb') as f:
    pickle.dump(val_seqs, f)

with open(PROCESSED / 'test_seqs.pkl', 'wb') as f:
    pickle.dump(test_seqs, f)

with open(PROCESSED / 'coldstart.pkl', 'wb') as f:
    pickle.dump(coldstart, f)

# Cũng lưu flat CSV với đầy đủ features
cols_keep = [
    'user_id', 'user_idx', 'question_id', 'q_idx',
    'competency', 'kc_idx', 'difficulty_label', 'diff_idx', 'difficulty',
    'quality_score', 'correct', 'quality_norm',
    'skill_before', 'skill_after', 'session_order', 'split'
]
log[cols_keep].to_csv(PROCESSED / 'interactions_featured.csv', index=False)

print('Saved:')
for f in sorted(PROCESSED.iterdir()):
    print(f'  {f.name} ({f.stat().st_size/1024:.1f} KB)')

---
## 8. Verify

In [ ]:
# Reload và kiểm tra
with open(PROCESSED / 'train_seqs.pkl', 'rb') as f:
    t = pickle.load(f)
with open(PROCESSED / 'metadata.json') as f:
    m = json.load(f)

print('=== Verify ===')
print(f'train_seqs loaded: {len(t)} sequences')
print(f'n_kcs={m["n_kcs"]}, n_users={m["n_users"]}')
print(f'kc_ids: {m["kc_ids"]}')
print()

# Kiểm tra correct_rate per split
for split_name, seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    all_correct = [c for s in seqs for c in s['correct_seq']]
    print(f'{split_name}: {len(seqs)} users, '
          f'{len(all_correct)} interactions, '
          f'correct_rate={np.mean(all_correct):.3f}')

print()
print('NEXT -> 03_train_models.ipynb')
print('  Input: data/processed/train_seqs.pkl, val_seqs.pkl, metadata.json')
print('  Models: BKT, DKT-Binary, DKT-Quality, SAKT-Binary, SAKT-Quality')